# DantinoX Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/01_quickstart.ipynb)

Covers:
- **Level-1** one-liner API (`dx.fit`, `dx.quick_generate`)
- **Level-2** explicit API (`ARParadigm`, `Trainer`, `ModelConfig`)
- Attention variants: MHA · GQA · MLA · Sliding-Window
- FFN variants: SwiGLU · GELU-MLP · Mixture-of-Experts
- Norm types: RMSNorm · LayerNorm
- Positional encodings: RoPE · learned · sinusoidal · none
- Text generation with `Generator` (greedy / top-k / nucleus / streaming)

**Runtime**: GPU (T4 or better recommended)

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import jax
print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt')
    print('Downloaded tiny_shakespeare.txt')
else:
    print('tiny_shakespeare.txt already present')

## Level 1 — One-liner API

`dx.fit('ar', corpus, **hyperparams)` trains a character-level AR model and returns the run directory.

In [ ]:
import dantinox as dx

run_dir = dx.fit(
    'ar', 'tiny_shakespeare.txt',
    dim=256, n_heads=4, head_size=64, num_blocks=4, vocab_size=200,
    lr=3e-4, epochs=2, batch_size=16,
)
print('Checkpoint saved to:', run_dir)

In [ ]:
output = dx.quick_generate(run_dir, 'HAMLET:\n', max_new_tokens=200)
print(output)

## Level 2 — Explicit Paradigm API

Separate `ModelConfig` (architecture) and `TrainingConfig` (training) for full control.

In [ ]:
model_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200, causal=True,
)
train_cfg = dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16)

paradigm = dx.ARParadigm(model_cfg)
run_dir2 = dx.Trainer(paradigm, train_cfg).fit('tiny_shakespeare.txt')
print('Run dir:', run_dir2)

## Attention Variants

| `attention=` | Description |
|---|---|
| `"mha"` | Multi-Head Attention (default) |
| `"gqa"` | Grouped-Query Attention — add `kv_heads < n_heads` |
| `"mla"` | Multi-Latent Attention (DeepSeek-V2 style) |
| `"mha"` + `sliding_window=True` | Local Sliding-Window Attention |

In [ ]:
from flax import nnx
import jax

def param_count(cfg):
    m = dx.ARParadigm(cfg).build_model(nnx.Rngs(0))
    return sum(x.size for x in jax.tree_util.tree_leaves(nnx.state(m, nnx.Param)))

attn_configs = [
    ('MHA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha')),
    ('GQA kv_heads=2',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=2)),
    ('GQA kv_heads=1',   dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='gqa', kv_heads=1)),
    ('MLA',               dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mla')),
    ('SWA window=64',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                         vocab_size=200, attention='mha',
                                         sliding_window=True, context_window=64)),
]

for name, cfg in attn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

In [ ]:
# Train with GQA — fewer KV heads means faster inference and lower KV-cache memory
gqa_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200, attention='gqa', kv_heads=2,
)
gqa_run = dx.Trainer(
    dx.ARParadigm(gqa_cfg),
    dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16),
).fit('tiny_shakespeare.txt', run_dir='/tmp/dx_gqa')
print(dx.quick_generate(gqa_run, 'HAMLET:\n', max_new_tokens=100))

## FFN Variants

| `ffn=` | Description |
|---|---|
| `"mlp"` | SwiGLU MLP (default) |
| `"mlp"` + `ffn_activation="gelu"` | GELU-activated MLP |
| `"moe"` | Mixture-of-Experts — add `n_experts` and `top_k` |

In [ ]:
ffn_configs = [
    ('SwiGLU MLP',     dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp')),
    ('GELU MLP',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='mlp', ffn_activation='gelu')),
    ('MoE 4 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=4, top_k=2)),
    ('MoE 8 experts',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=200, ffn='moe', n_experts=8, top_k=2)),
]

for name, cfg in ffn_configs:
    n = param_count(cfg)
    print(f'{name:20s}  {n/1e6:.2f}M params')

## Norm & Positional Encoding Variants

| `norm=` | `pos_encoding=` | Notes |
|---|---|---|
| `"rmsnorm"` | `"rotary"` | Default — RoPE is relative, no pos embedding matrix |
| `"layernorm"` | `"learned"` | Classic BERT-style learned absolute positions |
| `"rmsnorm"` | `"absolute"` | Fixed sinusoidal (Transformer 2017) |
| `"rmsnorm"` | `"none"` | No position info — rely on attention patterns only |

In [ ]:
norm_pos_configs = [
    ('RMSNorm + RoPE',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='rotary')),
    ('LayerNorm + Learned',  dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='layernorm', pos_encoding='learned')),
    ('RMSNorm + Sinusoidal', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='absolute')),
    ('RMSNorm + None',       dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                             vocab_size=200, norm='rmsnorm', pos_encoding='none')),
]

for name, cfg in norm_pos_configs:
    n = param_count(cfg)
    print(f'{name:30s}  {n/1e6:.2f}M params')

In [ ]:
# Compare RMSNorm vs LayerNorm on the same task
for norm in ('rmsnorm', 'layernorm'):
    cfg = dx.ModelConfig(dim=128, n_heads=2, head_size=64, num_blocks=4,
                         vocab_size=200, norm=norm)
    rd = dx.Trainer(
        dx.ARParadigm(cfg),
        dx.TrainingConfig(lr=3e-4, epochs=1, batch_size=16),
    ).fit('tiny_shakespeare.txt', run_dir=f'/tmp/dx_{norm}')
    print(f'{norm}: {dx.quick_generate(rd, "HAMLET:", max_new_tokens=60)[:80]}')

## Generator — Greedy / Top-k / Nucleus / Streaming

`Generator` wraps any trained model with multiple decoding strategies.

In [ ]:
from dantinox.generator import Generator

model = dx.load(run_dir)         # load from run_dir (Level-1 run above)
gen   = Generator(model, tokenizer_type='char')

for label, kwargs in [
    ('Greedy',          dict(temperature=0.0)),
    ('Top-k (k=40)',    dict(temperature=0.8, top_k=40)),
    ('Nucleus (p=0.9)', dict(temperature=0.9, top_p=0.9)),
]:
    print(f'=== {label} ===')
    print(gen.generate('HAMLET:\n', max_new_tokens=80, **kwargs))
    print()

In [ ]:
# Streaming — yields tokens one at a time
print('=== Streaming (top-k, k=50) ===')
for chunk in gen.stream('HAMLET:\n', max_new_tokens=80, temperature=0.8, top_k=50):
    print(chunk, end='', flush=True)
print()

## Analytical FLOPs Profile

`dx.profile` returns FLOPs breakdown without running any training.

In [ ]:
for dim, blocks in [(128, 4), (256, 8), (512, 12), (768, 24)]:
    cfg   = dx.ModelConfig(dim=dim, n_heads=max(1, dim//64), head_size=64,
                           num_blocks=blocks, vocab_size=200)
    flops = dx.profile(cfg, seq_len=256, batch_size=4)
    n     = param_count(cfg)
    print(f'dim={dim:4d} blocks={blocks:2d}  {n/1e6:6.1f}M params  {flops.total/1e9:.2f} GFLOPs')